In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split

In [3]:
df = pd.read_csv("Date_fruit.csv")
df.head()
df.shape
df["Class"].unique()

array(['BERHI', 'DEGLET', 'DOKOL', 'IRAQI', 'ROTANA', 'SAFAVI', 'SOGAY'],
      dtype=object)

In [4]:
X = df.drop("Class", axis = 1)
y = df["Class"]
le = LabelEncoder()
y = le.fit_transform(y)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

In [5]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# ANN

In [6]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

In [7]:
X_train_tensor = torch.tensor(X_train_scaled, dtype = torch.float32)
y_train_tensor = torch.tensor(y_train, dtype = torch.long)

X_test_tensor = torch.tensor(X_test_scaled, dtype = torch.float32)
y_test_tensor = torch.tensor(y_test, dtype = torch.long)

In [9]:
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

In [10]:
train_loader = DataLoader(train_dataset, batch_size = 32, shuffle = True)
test_loader = DataLoader(test_dataset, batch_size = 32)

In [15]:
#building model

class ANN(nn.Module):
    def __init__(self):
        super(ANN, self).__init__()

        self.model = nn.Sequential(
            nn.Linear(X.shape[1], 64),
            nn.ReLU(),

            nn.Linear(64,64),
            nn.ReLU(),

            nn.Linear(64, 7)
        )
    def forward(self, x):
        return self.model(x)

In [16]:
model = ANN()

#loss and optimizer
criteria = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

In [19]:
# training the NN

epochs = 100
for epoch in range(epochs):
    model.train()

    running_loss = 0.0
    for xb,yb in train_loader:
        optimizer.zero_grad()
        outputs = model(xb)
        loss = criteria(outputs, yb)
        loss.backward()
        optimizer.step() # params update

        running_loss += loss.item()
    train_loss = running_loss / len(train_loader)
    print(f"epoch is {epoch + 1}/{epochs} and loss is {train_loss}")

epoch is 1/100 and loss is 0.024717617010616737
epoch is 2/100 and loss is 0.023704796563833952
epoch is 3/100 and loss is 0.023679714447454266
epoch is 4/100 and loss is 0.023005943107621177
epoch is 5/100 and loss is 0.023377866686686226
epoch is 6/100 and loss is 0.024430192996869268
epoch is 7/100 and loss is 0.02115079595039234
epoch is 8/100 and loss is 0.020506793453418853
epoch is 9/100 and loss is 0.02083302076663012
epoch is 10/100 and loss is 0.01992208889240156
epoch is 11/100 and loss is 0.021218069297585473
epoch is 12/100 and loss is 0.019956872960471588
epoch is 13/100 and loss is 0.01801143338620339
epoch is 14/100 and loss is 0.02287675551665218
epoch is 15/100 and loss is 0.019923599320463836
epoch is 16/100 and loss is 0.019069050843624966
epoch is 17/100 and loss is 0.020356969806649115
epoch is 18/100 and loss is 0.0173171406090462
epoch is 19/100 and loss is 0.01651308770574953
epoch is 20/100 and loss is 0.016029028611703088
epoch is 21/100 and loss is 0.0153144

In [22]:
# Evaluation

model.eval()

total = 0
correct = 0

with torch.no_grad():
    for xb, yb in test_loader:
        outputs = model(xb)
        max_val, predicted = torch.max(outputs, 1)

        correct += (predicted == yb).sum().item()
        total += yb.size(0)
print(f"total values {total}")
print(f"correct values {correct}")
print(f"Accuracy is {correct/total}")

total values 180
correct values 166
Accuracy is 0.9222222222222223
